[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WesamKhallaf73/pytorch-ground-up-notebooks/blob/main/notebooks/L15.ipynb)

# Lecture 15 — Splitting Data & the Anatomy of a Training Loop

*Three splits that keep you honest about how well your model really works — and the seven lines that make it learn.*

**PyTorch from Ground Up · Volume I · Chapters 26–27** — companion notebook.

**How to use it:** run the cells from top to bottom. Before each code cell, *predict* what it will print — then run it and compare with the expected output (click ▸ to open it). Every exercise from the lecture is at the end.

By the end you will be able to:
- Explain why you need three splits, and what each one is for.
- Split reproducibly with `random_split` and a seeded Generator.
- Recognise the three classic data leaks and the split-first rule.
- Choose stratified, chronological or pre-split strategies when random splitting is wrong.
- Write the canonical seven-line loop from memory and explain every line.
- Count epochs, batches and updates, track the true epoch loss, and add a validation phase.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
print('PyTorch', torch.__version__)

## 01 · Why split — and why three splits

Train on all your data and evaluate on the same data, and you are **grading your own homework**.

<sub>Source: lecture slides 3–8</sub>

> ⚠️ **Watch out — The test set is sacred**  
> Look at it once, report the number, and stop.
>
> If you go back and change your model based on the test score, then re-evaluate, you are no longer testing — you are tuning, and your “test” score is a lie.

## 02 · random_split and reproducible splits

`random_split` returns `Subset`s — **views, not copies**, and still Datasets. Each goes straight into its own DataLoader. Only the training loader shuffles.

<sub>Source: lecture slides 9–14</sub>

**`split.py`** — predict the output, then run.

In [ ]:
from torch.utils.data import random_split, TensorDataset, DataLoader
# 1000 samples
x = torch.randn(1000, 10)
y = torch.randint(0, 3, (1000,))
full_ds = TensorDataset(x, y)
# Split: 700 train, 150 val, 150 test
gen = torch.Generator().manual_seed(42)
train_ds, val_ds, test_ds = random_split(
    full_ds, [700, 150, 150],
    generator=gen)
print(len(train_ds), len(val_ds), len(test_ds))

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False)

<details><summary>▸ Expected output</summary>

```
700 150 150
```
</details>

🧪 *Interactive lab in the app:* **Three piles — and how to cut them** — try it there to see this idea move.

## 03 · Data leakage

**Data leakage**: information from val or test **bleeding into training**. It makes the model look better than it is — the most common reason models shine in the lab and fail in the world.

<sub>Source: lecture slides 15–21</sub>

**`leakage.py`** — a pattern to recognise (the `...` stand for your own code), not a cell to run:

```python
# ✗ Wrong: normalize on full data, then split
x_all = (x_all - x_all.mean(0)) / x_all.std(0)
train, val, test = random_split(...)

# ✓ Correct: split first, normalize from train only
train, val, test = random_split(...)
mean = x[train.indices].mean(0)
std  = x[train.indices].std(0)
# Apply mean, std to all three splits
```

> ⚠️ **Watch out — Leakage makes your metrics better**  
> Leakage is insidious because it makes your metrics **better**, not worse. You will not notice it from the numbers alone — only when the model fails in production.
>
> Always split first, then preprocess. Never compute statistics on the full dataset.

## 04 · Stratified, time-series and pre-split datasets

Random splits can **miss a rare class entirely**. A stratified split keeps the proportions. If class A is 5% of the data, a random 15% split may grab most of it — or none. PyTorch has no stratified split built in; scikit-learn does.

<sub>Source: lecture slides 22–33</sub>

**`stratified.py`** — predict the output, then run.

In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset
indices = list(range(len(full_ds)))
labels  = y.numpy()
# First split: train+val vs test
trainval_idx, test_idx = train_test_split(
    indices, test_size=0.15, stratify=labels,
    random_state=42)
# Second split: train vs val
trainval_labels = labels[trainval_idx]
train_idx, val_idx = train_test_split(
    trainval_idx, test_size=0.176,  # 0.176 of 85% ≈ 15%
    stratify=trainval_labels, random_state=42)
train_ds = Subset(full_ds, train_idx)
val_ds   = Subset(full_ds, val_idx)
test_ds  = Subset(full_ds, test_idx)
print(len(train_ds), len(val_ds), len(test_ds))

<details><summary>▸ Expected output</summary>

```
700 150 150
```
</details>

**`time_split.py`** — predict the output, then run.

In [ ]:
# Time series: first 70% train, next 15% val, last 15% test
n = len(full_ds)
n_train = int(0.70 * n)
n_val   = int(0.15 * n)
train_ds = Subset(full_ds, range(n_train))
val_ds   = Subset(full_ds, range(n_train, n_train + n_val))
test_ds  = Subset(full_ds, range(n_train + n_val, n))

## 05 · The canonical loop, line by line

Part V: everything you’ve learned meets in **one pattern — the training loop**. Tensors, autograd, losses, optimizers, DataLoaders: the single most important pattern in deep learning.

<sub>Source: lecture slides 35–42, 55</sub>

_Stand-in: the next listing uses names from your own project (a model, a data loader…). This cell defines small ones so it runs here._

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
train_loader = DataLoader(TensorDataset(torch.randn(256, 10), torch.randn(256, 1)), batch_size=32, shuffle=True)
model = nn.Linear(10, 1)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
num_epochs = 3


**`loop.py`** — predict the output, then run.

In [ ]:
for epoch in range(num_epochs):             # ❶
    for batch_x, batch_y in train_loader:   # ❷
        pred = model(batch_x)               # ❸
        loss = loss_fn(pred, batch_y)       # ❹
        optimizer.zero_grad()               # ❺
        loss.backward()                     # ❻
        optimizer.step()                    # ❼

## 06 · Epochs, batches and a complete run

Report the **true epoch average**: sum `loss × batch size`, divide by sample count. That stays honest when the last batch is short — last lecture printed only the final batch’s loss.

<sub>Source: lecture slides 43–52</sub>

🧪 *Interactive lab in the app:* **Counting updates** — try it there to see this idea move.

**`train_example.py`** — predict the output, then run.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
torch.manual_seed(0)
x = torch.randn(500, 10)
w_true = torch.randn(10, 1)
y = x @ w_true + 0.5 * torch.randn(500, 1)
ds = TensorDataset(x, y)
loader = DataLoader(ds, batch_size=32, shuffle=True)

model   = nn.Linear(10, 1)
loss_fn = nn.MSELoss()
opt     = torch.optim.Adam(model.parameters(), lr=1e-2)

losses = []
for epoch in range(30):
    epoch_loss = 0.0
    for bx, by in loader:
        pred = model(bx)
        loss = loss_fn(pred, by)
        opt.zero_grad()
        loss.backward()
        opt.step()
        epoch_loss += loss.item() * bx.size(0)
    avg_loss = epoch_loss / len(ds)
    losses.append(avg_loss)
    if epoch % 5 == 0:
        print(f"Epoch {epoch:2d}  Loss: {avg_loss:.4f}")

<details><summary>▸ Expected output</summary>

```
Epoch  0  Loss: 19.4447
Epoch  5  Loss: 8.8616
Epoch 10  Loss: 4.0405
Epoch 15  Loss: 1.8793
Epoch 20  Loss: 0.9249
Epoch 25  Loss: 0.5113
```
</details>

## 07 · What breaks if you remove a line



<sub>Source: lecture slides 53–54</sub>

🧪 *Interactive lab in the app:* **Five ways to break the loop** — try it there to see this idea move.

## 08 · Adding validation

Training loss measures **fit**, not **generalisation**. Check validation after every epoch.

<sub>Source: lecture slides 56–59</sub>

_Stand-in: the next listing uses names from your own project (a model, a data loader…). This cell defines small ones so it runs here._

In [ ]:
num_epochs = 3
train_loader = loader
val_loader = DataLoader(TensorDataset(torch.randn(100, 10), torch.randn(100, 1)), batch_size=32)


**`train_val.py`** — predict the output, then run.

In [ ]:
for epoch in range(num_epochs):
    # — Training ——————————————
    model.train()
    for bx, by in train_loader:
        pred = model(bx)
        loss = loss_fn(pred, by)
        opt.zero_grad()
        loss.backward()
        opt.step()
    # — Validation ————————————
    model.eval()
    with torch.no_grad():
        val_loss = 0.0
        for bx, by in val_loader:
            pred = model(bx)
            val_loss += loss_fn(pred, by).item()
    print(f"Val loss: {val_loss / len(val_loader):.4f}")

## The rule — worth remembering

**Splitting — seven rules**

1. Three splits: train (learn), validation (tune), test (report). The test set is touched once.
2. Use `random_split` with a fixed Generator seed for reproducibility.
3. Split first, preprocess second. Compute normalization statistics on the training set only.
4. Data leakage makes metrics look better — but the model fails on real data. Always be suspicious of unrealistically good results.
5. Use stratified splits for imbalanced data to preserve class proportions.
6. For time-series: split chronologically, never randomly.
7. Respect pre-split benchmarks. Carve a validation set from the training split if needed.

**The training loop — six rules**

1. The training loop has two nested loops: epochs (outer) and batches (inner).
2. Inside the batch loop: predict → loss → zero_grad → backward → step. Always this order.
3. Track the average epoch loss to monitor convergence.
4. A healthy loss curve drops steeply then plateaus. Exploding or flat curves mean something is wrong.
5. Every line in the loop exists for a reason. Removing or reordering any one breaks training.
6. Add a validation loop after each training epoch to monitor generalization.

## Pause and try

Work each exercise in the empty cell below it. Open the worked answer only after you have tried.

### Exercise 01 · Try it — same split twice?

Run `random_split` twice without a generator and print the first sample of each resulting training set. Are they different? Now add a fixed Generator and try again. Do they match?

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

Without a generator the two training sets start with different samples. With `torch.Generator().manual_seed(42)` created fresh for each call, they match exactly. (Reusing one generator object for a second call in the same run gives a different split — it has advanced.)
</details>

### Exercise 02 · Try it — random vs stratified

Create a dataset with 90% class 0 and 10% class 1 (1,000 samples). Split it randomly and count the class distribution in each split. Then split with stratification and compare.

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

A random split gives roughly — but not exactly — 10% in each split, and the small splits wobble most (a 150-sample validation set might hold 9 or 21 positives). Stratified gives 70 / 15 / 15 positives: exactly 10% everywhere. The split lab above shows it.
</details>

### Exercise 03 · Ch 26 · Ex 4

Simulate 365 daily measurements. Split chronologically: train = first 250, val = next 50, test = last 65. Why would random splitting be wrong here?

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

`Subset(ds, range(250))`, `range(250, 300)`, `range(300, 365)`. A random split would train on days that come *after* the ones being predicted, letting the model interpolate between neighbours — a score it could never reproduce on genuinely future data.
</details>

### Exercise 04 · Ch 26 · Ex 6

Deliberately leak: normalize the full dataset, split, train. Then do it correctly (normalize from training only). Compare validation accuracy. Is the leaked version higher? What does that tell you?

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

The leaked version is typically equal or slightly higher. That is the lesson: leakage flatters your metrics, so you cannot detect it from the numbers — only by auditing the pipeline.
</details>

### Exercise 05 · Ch 27 · Ex 1, 3, 4

Type the canonical loop from memory. Then remove `optimizer.zero_grad()` and train 20 epochs printing the loss — what happens? Then swap `backward()` and `step()` — does the model still learn?

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

Without `zero_grad()` the loss stops falling and swings or blows up as the accumulated gradients grow. With `backward()` and `step()` swapped the loss is flat: every `step()` sees freshly zeroed gradients. Both are in the loop-breaker lab.
</details>

### Exercise 06 · Ch 27 · Ex 7 and plotting

Plot `losses` with matplotlib; try lr = 1e-1 and 1e-4. Then try batch size 1 (pure SGD) and 500 (full-batch GD). How does the loss curve change? Which converges fastest?

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

lr = 1e-1 falls much faster (and may wobble); lr = 1e-4 barely moves in 30 epochs. Batch size 1 makes 500 noisy updates per epoch and converges in the fewest epochs; batch size 500 makes one smooth update per epoch and needs many more epochs.
</details>

### Interview-style

(a) “What is data leakage? Give two concrete examples. Why three splits instead of two?” — five sentences total.

_Write your answer here before opening the model answer._

<details><summary>▸ Model answer</summary>

Data leakage is information from the validation or test data reaching the training process, which makes the model look better than it really is. One example is computing normalization statistics on the full dataset before splitting. Another is the same or near-duplicate sample landing in both training and test sets — or, for time series, a random split that lets the model see the future. We need three splits because decisions made during training (learning rate, stopping epoch, architecture) are themselves fitted to whatever data guides them. So validation guides those decisions, and a separate test set — touched once — gives the only unbiased estimate.
</details>

### Interview-style

(b) “Write a PyTorch training loop on the whiteboard and explain each line” — all seven lines, in order, one sentence each.

_Write your answer here before opening the model answer._

<details><summary>▸ Model answer</summary>

`for epoch in range(num_epochs):` — repeat full passes over the training data. `for batch_x, batch_y in train_loader:` — take one mini-batch at a time. `pred = model(batch_x)` — forward pass; builds the computation graph. `loss = loss_fn(pred, batch_y)` — measure the error as one number. `optimizer.zero_grad()` — clear the gradients left from the previous batch. `loss.backward()` — compute the gradient of the loss for every parameter. `optimizer.step()` — update every parameter using its gradient.
</details>